In [ ]:
print("Hotel_Success_Prediction");

In [ ]:
!pip install numpy pandas scikit-learn xgboost tensorflow shap

Importing Libraries

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import xgboost as xgb
import shap
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge, LogisticRegression

Setting seed

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

Loading the data set

In [ ]:
drive.mount('/content/drive')

print("Reading the dataset from my Google Drive")
df = pd.read_csv('/content/drive/MyDrive/Hotel_Reviews.csv')
print(f"Dataset successfully loaded. Shape: {df.shape}")

In [ ]:
df.head()


In [ ]:
df.columns

Combining Positive_Review and Negative_Review

In [ ]:
df["Negative_Review"] = df["Negative_Review"].fillna("")
df["Positive_Review"] = df["Positive_Review"].fillna("")

df["Review_Text"] = df["Negative_Review"].astype(str) + " " + df["Positive_Review"].astype(str)

In [ ]:
df.columns

In [ ]:
df.head()

Creating the hotel label with Average Score

In [ ]:
df["Hotel_Label"] = (df["Average_Score"] >= 8.0).astype(int)



In [ ]:
df[["Hotel_Name", "Average_Score", "Reviewer_Score", "Hotel_Label", "Review_Text"]].head()

Hotel wise split for training and test datasets

In [ ]:
hotel_labels = df.groupby("Hotel_Name")["Hotel_Label"].first().reset_index()

train_hotels, test_hotels = train_test_split(
    hotel_labels,
    test_size=0.2,
    random_state=42,
    stratify=hotel_labels["Hotel_Label"]
)

train_hotel_names = set(train_hotels["Hotel_Name"])
test_hotel_names = set(test_hotels["Hotel_Name"])

train_df = df[df["Hotel_Name"].isin(train_hotel_names)].copy()
test_df = df[df["Hotel_Name"].isin(test_hotel_names)].copy()

print(train_df["Hotel_Name"].nunique(), test_df["Hotel_Name"].nunique())
print(set(train_df["Hotel_Name"]).intersection(set(test_df["Hotel_Name"])))

In [ ]:
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

In [ ]:
train_df.head()

In [ ]:
test_df.head()

In [ ]:
print("Train Hotels:", train_df["Hotel_Name"].nunique())
print("Test Hotels:", test_df["Hotel_Name"].nunique())

Checking if the split is correctly done

In [ ]:
common_hotels = set(train_df["Hotel_Name"]).intersection(set(test_df["Hotel_Name"]))

print("Common Hotels:", len(common_hotels))

Checking number of hotels according to their label.

In [ ]:
print("Training Set")
print(train_df.groupby("Hotel_Name")["Hotel_Label"].first().value_counts())

In [ ]:
print("Test set")
print(test_df.groupby("Hotel_Name")["Hotel_Label"].first().value_counts())

Review Text and Rating incosistency filter

Training the ridge ob train data set



In [ ]:
ridge_tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    stop_words="english"
)

X_train_text_ridge = ridge_tfidf.fit_transform(train_df["Review_Text"])
y_train_score = train_df["Reviewer_Score"]

ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_text_ridge, y_train_score)

Predicting the ratings based on text review

In [ ]:
train_df["Predicted_Text_Score"] = ridge_model.predict(X_train_text_ridge)

X_test_text_ridge = ridge_tfidf.transform(test_df["Review_Text"])
test_df["Predicted_Text_Score"] = ridge_model.predict(X_test_text_ridge)

Calculating the inconsistency

In [ ]:
train_df["Delta"] = abs(train_df["Reviewer_Score"] - train_df["Predicted_Text_Score"])
test_df["Delta"] = abs(test_df["Reviewer_Score"] - test_df["Predicted_Text_Score"])

threshold = 2.0

train_clean = train_df[train_df["Delta"] < threshold].copy()
test_clean = test_df[test_df["Delta"] < threshold].copy()

print("Before filtering:", train_df.shape, test_df.shape)
print("After filtering:", train_clean.shape, test_clean.shape)